In [1]:
import functools
from src.inference.vllm_inference import setup_model
from src.data.loader import load_benchmark
from src.evaluator import Evaluator
from src.metrics import PassAtk, GreedyPass, PercentPassed, MeanEntropy
from src.data.prompt import mbpp, humaneval
import src.config as config
import datasets

import gc
import os
import torch
from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment

In [4]:
def run_experiments_benchmarks(experiments):

	raw_mbpp = datasets.load_dataset("google-research-datasets/mbpp", "sanitized", split="test")
	raw_he = datasets.load_dataset("openai/openai_humaneval", split="test")

	metrics = [
		GreedyPass(),
		PassAtk(k=1, n_samples=1),
		# PassAtk(k=10, n_samples=10), # слишком жирная операция
		PercentPassed(),
		MeanEntropy(),
	]

	for i, (exp_name, model_path, adapter_path) in enumerate(experiments):
		print(f"=== Running experiments #{i+1}/{len(experiments)}: {exp_name} ===")
		llm, tokenizer, _ = setup_model(model_path=model_path, adapter_path=adapter_path)

		mbpp_mapper = functools.partial(mbpp.mbpp_to_task, tokenizer=tokenizer)
		he_mapper = functools.partial(humaneval.humaneval_to_task, tokenizer=tokenizer)

		tasks_mbpp = load_benchmark(raw_mbpp, mbpp_mapper)
		tasks_he = load_benchmark(raw_he, he_mapper)

		evaluator = Evaluator(llm, tokenizer, metrics)

		print("=== MBPP RESULTS ===")
		print(evaluator.run(tasks_mbpp))

		print("=== HUMANEVAL RESULTS ===")
		print(evaluator.run(tasks_he))

		# Секция клининга
		del llm, tokenizer, evaluator
		del tasks_mbpp, tasks_he, mbpp_mapper, he_mapper

		destroy_model_parallel()
		destroy_distributed_environment()

		gc.collect()
		if torch.cuda.is_available():
			torch.cuda.empty_cache()
			torch.cuda.ipc_collect()

In [3]:
def prepare_experiments():
    experiments = []
    models_dir = config.MODELS_DIR
    folder_names = sorted(os.listdir(models_dir))

    for folder_name in folder_names:
        full_path = os.path.join(models_dir, folder_name)

        if not os.path.isdir(full_path) or folder_name == "cache" or folder_name.startswith("."):
            continue

        experiment_name = folder_name
        if "-sft-" in folder_name or "-sgd-" in folder_name:
            separator = "-sft-" if "-sft-" in folder_name else "-sgd-"

            base_model_name = folder_name.split(separator)[0]
            model_path = os.path.join(models_dir, base_model_name)
            adapter_path = full_path
        else:
            model_path = full_path
            adapter_path = None

        experiments.append((experiment_name, model_path, adapter_path))

    return experiments

In [6]:
experiments = prepare_experiments()
experiments

[('qwen3-0.6b', './models/qwen3-0.6b', None),
 ('qwen3-0.6b-sft-1-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sft-1-exp'),
 ('qwen3-0.6b-sft-3-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sft-3-exp'),
 ('qwen3-0.6b-sft-5-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sft-5-exp'),
 ('qwen3-0.6b-sft-7-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sft-7-exp'),
 ('qwen3-0.6b-sgd-2-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sgd-2-exp'),
 ('qwen3-0.6b-sgd-4-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sgd-4-exp'),
 ('qwen3-0.6b-sgd-6-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sgd-6-exp'),
 ('qwen3-0.6b-sgd-8-exp',
  './models/qwen3-0.6b',
  './models/qwen3-0.6b-sgd-8-exp'),
 ('qwen3-1.7b', './models/qwen3-1.7b', None),
 ('qwen3-1.7b-sft-1-exp',
  './models/qwen3-1.7b',
  './models/qwen3-1.7b-sft-1-exp'),
 ('qwen3-1.7b-sft-3-exp',
  './models/qwen3-1.7b',
  './models/qwen3-1.7b-sft-3-exp'),
 ('qwen3-1.7b-sft-5-exp',
  './models/

In [7]:
run_experiments_benchmarks(experiments)

=== Running experiments #1/17: qwen3-0.6b ===
INFO 02-21 20:23:17 [utils.py:261] non-default args: {'trust_remote_code': True, 'seed': 42, 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'max_lora_rank': None, 'model': './models/qwen3-0.6b'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-21 20:23:17 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-21 20:23:17 [model.py:1561] Using max model len 4096
INFO 02-21 20:23:17 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-21 20:23:17 [vllm.py:624] Asynchronous scheduling is enabled.
WARNING 02-21 20:23:18 [interface.py:470] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
WARNING 02-21 20:23:18 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: WSL is detected and NVML is not compatible with fork
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:25 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='./models/qwen3-0.6b', speculative_config=None, tokenizer='./models/qwen3-0.6b', skip_tokenizer_init=False, tokenizer_mode=auto, revis

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.70s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.70s/it]
(EngineCore_DP0 pid=3171) 


(EngineCore_DP0 pid=3171) INFO 02-21 20:23:31 [default_loader.py:291] Loading weights took 1.73 seconds
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:31 [gpu_model_runner.py:4130] Model loading took 1.12 GiB memory and 2.852789 seconds
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:37 [backends.py:812] Using cache directory: /root/.cache/vllm/torch_compile_cache/8bf3928dc7/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:37 [backends.py:872] Dynamo bytecode transform time: 5.21 s
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:45 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 3.688 s
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:45 [monitor.py:34] torch.compile takes 8.90 s in total
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:46 [gpu_worker.py:356] Available KV cache memory: 8.64 GiB
(EngineCore_DP0 pid=3171) INFO 02-21 20:23:46 [kv_cache_utils.py:1307] GPU KV cache size: 80,928 tokens
(EngineCore_DP0 pid=3171

...

Processed prompts:   0%|          | 0/164 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

🚀 Executing 164 samples in parallel using 8 workers...


Running Tests:   0%|          | 0/164 [00:00<?, ?it/s]

📊 pass@1 (n=1): 0.6646
📊 mean_%passed: 0.7086
📊 mean_entropy: 0.1126
{'greedy@1': 0.6829268292682927, 'pass@1 (n=1)': np.float64(0.6646341463414634), 'mean_%passed': np.float64(0.7086164343786295), 'mean_entropy': np.float64(0.11261360722765205)}


[rank0]:[W222 01:10:29.954546981 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
